# 05 — Assemble training panel

Join the existing PIT panel (CRSP + Sharadar) with macro indicators, aux text
features (`text_novelty`, `days_since_filing`, `doc_count_7d`), and forward-return
labels. Output: `data/processed/training_panel/year=YYYY/part-0.parquet`,
daily cadence, partitioned by year. Consumed by notebook 06 (ranker,
Friday-filtered) and notebook 07+ (RL agent, daily).

**Spec:** `docs/superpowers/specs/2026-05-16-feature-assembly-design.md`.

**Resumable:** existing year-shards are skipped on re-run. Delete a year's
`part-0.parquet` to re-do.

PCA text features are *not* in this panel — they're derived at ranker time
from `finbert_stockday_embed` + the walk's `pca.joblib`. See spec §6.

## A. Setup

In [ ]:
from __future__ import annotations
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm

from src.utils.io import processed_dir, repo_root
from src.utils.features import (
    pivot_macro_wide,
    compute_forward_returns,
    compute_text_novelty,
    compute_days_since_filing,
    compute_doc_count_window,
)

PANEL_DIR = processed_dir() / 'panel'
MACRO_PATH = processed_dir() / 'macro.parquet'
EMBED_DIR = processed_dir() / 'finbert_stockday_embed'
EDGAR_INDEX_PATH = processed_dir() / 'edgar_index.parquet'
UNIVERSE_PATH = processed_dir() / 'universe_ids.parquet'
OUT_DIR = processed_dir() / 'training_panel'
SUMMARY_PATH = repo_root() / 'reports' / 'metrics' / 'feature_assembly_summary.json'

OUT_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)

# Validation thresholds (spec §8).
MAX_TEXT_NOVELTY_NAN_RATE = 0.05
MAX_DAYS_SINCE_FILING_NAN_RATE = 0.10

YEARS = sorted({int(p.name.split('=')[1]) for p in PANEL_DIR.glob('year=*')})
print(f'panel years: {YEARS[0]}..{YEARS[-1]} ({len(YEARS)} years)')
print(f'out_dir: {OUT_DIR}')

## B. Load shared inputs (universe, macro, filings)

Loaded once outside the per-year loop. Universe + filings are small; the
embed shards are accessed per-year inside the loop to keep peak memory bounded.

In [ ]:
universe_ids = pd.read_parquet(UNIVERSE_PATH)
universe_ids['permno'] = universe_ids['permno'].astype('Int64')
universe_ids['date_out'] = universe_ids['date_out'].fillna(pd.Timestamp('2099-12-31'))
print(f'universe_ids: {len(universe_ids)} rows, {universe_ids["permno"].nunique()} permnos')

macro_long = pd.read_parquet(MACRO_PATH)
print(f'macro: {len(macro_long)} rows, series={sorted(macro_long["series"].unique())}')

edgar_index = pd.read_parquet(EDGAR_INDEX_PATH)
edgar_index['filing_date'] = pd.to_datetime(edgar_index['filing_date'])
# Restrict to ciks in our universe — drops noise from the broader EDGAR pull.
edgar_index = edgar_index[edgar_index['cik'].isin(universe_ids['cik'])]
print(f'edgar_index (universe-filtered): {len(edgar_index)} filings')

## C. Per-year assembly helper

`assemble_year(year)` does: read `panel/year=YYYY/`, universe-filter, left-join
macro (pivoted + ffilled over that year's date range), left-join cik via
`universe_ids` (so we can attach filings later), then attach aux text features
(`text_novelty` from the year's embed shard; `days_since_filing` + `doc_count_7d`
from `edgar_index`). Returns the per-year dataframe ready for labels (cell D)
and persist (cell F).

In [ ]:
def _read_year_embed(year: int) -> pd.DataFrame:
    """Read finbert_stockday_embed shards for one year. Also peeks at year-1's
    last 7 days so the year's earliest rows can find a t-7 predecessor."""
    shards = sorted(EMBED_DIR.glob(f'year={year}/*.parquet'))
    prior_shards = sorted(EMBED_DIR.glob(f'year={year - 1}/*.parquet'))
    frames = []
    for s in shards:
        frames.append(pd.read_parquet(s, columns=['permno', 'date', 'vec']))
    if prior_shards:
        for s in prior_shards:
            df = pd.read_parquet(s, columns=['permno', 'date', 'vec'])
            df = df[df['date'] >= pd.Timestamp(f'{year - 1}-12-24')]  # last week of prior year
            frames.append(df)
    if not frames:
        return pd.DataFrame(columns=['permno', 'date', 'vec'])
    embed = pd.concat(frames, ignore_index=True)
    embed['date'] = pd.to_datetime(embed['date'])
    return embed


def assemble_year(year: int) -> pd.DataFrame:
    # 1. Read panel year.
    panel_shards = sorted(PANEL_DIR.glob(f'year={year}/*.parquet'))
    base = pd.concat([pd.read_parquet(s) for s in panel_shards], ignore_index=True)
    base['date'] = pd.to_datetime(base['date'])

    # 2. Universe filter (interval merge, same logic as src/utils/pca.py).
    intervals = universe_ids[['permno', 'cik', 'date_in', 'date_out']].copy()
    intervals['permno'] = intervals['permno'].astype('int64')
    merged = base.merge(intervals, on='permno', how='inner')
    in_window = (merged['date'] >= merged['date_in']) & (merged['date'] <= merged['date_out'])
    base = (merged[in_window]
            .drop(columns=['date_in', 'date_out'])
            .drop_duplicates(subset=['permno', 'date']))

    # 3. Macro: pivot, ffill to this year's calendar dates, left-join.
    year_dates = pd.DatetimeIndex(sorted(base['date'].unique()))
    macro_w = pivot_macro_wide(macro_long, ffill_dates=year_dates)
    base = base.merge(macro_w, on='date', how='left')

    # 4. Aux text features.
    embed = _read_year_embed(year)
    novelty = compute_text_novelty(embed, lookback_days=7)
    # Drop the borrowed t-1y rows from the output (we only kept them for lookups).
    novelty = novelty[novelty['date'].dt.year == year]
    base = base.merge(novelty, on=['permno', 'date'], how='left')

    dsf = compute_days_since_filing(edgar_index, base[['permno', 'cik', 'date']])
    base = base.merge(dsf, on=['permno', 'date'], how='left')

    docc = compute_doc_count_window(edgar_index, base[['permno', 'cik', 'date']], window_days=7)
    base = base.merge(docc, on=['permno', 'date'], how='left')

    return base

## D. Per-year label computation

Forward returns require lookahead, so labels for year Y need ret values from
year Y+1. `attach_labels` stitches a small prefix of year Y+1's panel onto
year Y before computing labels, then drops the prefix before returning.

In [ ]:
def attach_labels(year_df: pd.DataFrame, year: int, max_horizon: int = 5) -> pd.DataFrame:
    """Compute fwd_ret_1d and fwd_ret_5d for year_df, using year+1's prefix
    so the last `max_horizon` trading days of year_df get full labels."""
    next_year_shards = sorted(PANEL_DIR.glob(f'year={year + 1}/*.parquet'))
    if next_year_shards:
        prefix = pd.concat([pd.read_parquet(s) for s in next_year_shards], ignore_index=True)
        prefix['date'] = pd.to_datetime(prefix['date'])
        # Keep ~10 trading days (calendar buffer of 14) so the rolling has room.
        prefix = prefix[prefix['date'] <= pd.Timestamp(f'{year + 1}-01-15')]
        prefix = prefix.merge(
            universe_ids[['permno', 'date_in', 'date_out']],
            on='permno', how='inner',
        )
        in_window = (prefix['date'] >= prefix['date_in']) & (prefix['date'] <= prefix['date_out'])
        prefix = prefix[in_window].drop(columns=['date_in', 'date_out'])
        prefix = prefix[['permno', 'date', 'ret']].drop_duplicates(subset=['permno', 'date'])
        stitched = pd.concat(
            [year_df[['permno', 'date', 'ret']], prefix],
            ignore_index=True,
        )
    else:
        stitched = year_df[['permno', 'date', 'ret']]

    labeled = compute_forward_returns(stitched, horizons=(1, 5))
    labeled = labeled[labeled['date'].dt.year == year]
    return year_df.merge(
        labeled[['permno', 'date', 'fwd_ret_1d', 'fwd_ret_5d']],
        on=['permno', 'date'], how='left',
    )

## E. Validation gates

Per-year hard stops (spec §8). Raises on any gate failure — the year's
partial output is not persisted.

In [ ]:
def validate_year(df: pd.DataFrame, year: int) -> None:
    assert len(df) > 0, f'year {year}: empty dataframe'
    expected_cols = {
        'permno', 'date', 'ret', 'macro_vixcls', 'macro_dgs10', 'macro_dgs3mo',
        'macro_t10y2y', 'text_novelty', 'days_since_filing', 'doc_count_7d',
        'fwd_ret_1d', 'fwd_ret_5d',
    }
    missing = expected_cols - set(df.columns)
    assert not missing, f'year {year}: missing columns {missing}'

    novelty_nan = df['text_novelty'].isna().mean()
    assert novelty_nan <= MAX_TEXT_NOVELTY_NAN_RATE, (
        f'year {year}: text_novelty NaN rate {novelty_nan:.3f} '
        f'> threshold {MAX_TEXT_NOVELTY_NAN_RATE}'
    )

    dsf_nan = df['days_since_filing'].isna().mean()
    assert dsf_nan <= MAX_DAYS_SINCE_FILING_NAN_RATE, (
        f'year {year}: days_since_filing NaN rate {dsf_nan:.3f} '
        f'> threshold {MAX_DAYS_SINCE_FILING_NAN_RATE}'
    )

    # fwd_ret_5d NaN should be bounded — roughly: (last 5 trading days per permno)
    # / total rows. With ~250 trading days and ~600 permnos -> at most ~6 / 250
    # of rows. Use 5% as the ceiling.
    fwd_nan = df['fwd_ret_5d'].isna().mean()
    assert fwd_nan <= 0.05, (
        f'year {year}: fwd_ret_5d NaN rate {fwd_nan:.3f} > 0.05 '
        '(expected only tail rows of each permno)'
    )

## F. Per-year loop, atomic persist, summary

For each year, skip if `training_panel/year=YYYY/part-0.parquet` exists.
Otherwise: assemble → label → validate → atomic write (`*.tmp → rename`).
After the loop, write per-year row counts + NaN audit to
`reports/metrics/feature_assembly_summary.json`.

In [ ]:
def shard_path(year: int) -> Path:
    return OUT_DIR / f'year={year}' / 'part-0.parquet'


def process_year(year: int) -> dict:
    out_path = shard_path(year)
    if out_path.exists():
        n = pq.read_metadata(out_path).num_rows
        print(f'year={year}: exists ({n} rows) — skipping')
        return {'year': year, 'n_rows': int(n), 'status': 'skipped'}

    out_path.parent.mkdir(parents=True, exist_ok=True)
    df = assemble_year(year)
    df = attach_labels(df, year)
    validate_year(df, year)

    tmp = out_path.with_suffix('.parquet.tmp')
    df.to_parquet(tmp, compression='zstd', index=False)
    tmp.rename(out_path)
    print(f'year={year}: wrote {len(df)} rows -> {out_path}')
    return {
        'year': year,
        'n_rows': int(len(df)),
        'n_permnos': int(df['permno'].nunique()),
        'nan_rate_text_novelty': float(df['text_novelty'].isna().mean()),
        'nan_rate_days_since_filing': float(df['days_since_filing'].isna().mean()),
        'nan_rate_fwd_ret_5d': float(df['fwd_ret_5d'].isna().mean()),
        'status': 'written',
    }


per_year = [process_year(y) for y in YEARS]
SUMMARY_PATH.write_text(json.dumps({'years': per_year}, indent=2))
print(f'\nsummary -> {SUMMARY_PATH.relative_to(repo_root())}')